# Connecting to World Bank

In [1]:
import requests
import pandas as pd

world_bank_url = "https://api.worldbank.org/v2/country"
world_bank_response = requests.get(world_bank_url)                # 2A- Send the request and save the response

if world_bank_response.status_code == 200:
    print("Connection succesful")
else:
    print("Connection not succesful")

Connection succesful


In [2]:
# Countries using ISO codes (CHE = Switzerland, GBR = United Kingdom)
countries = "AT;BE;FR;DE;IT;NL;PL;ES;CHE;GBR"

# World Bank indicator codes mapped to readable names
INDICATORS = {

    "population":        "SP.POP.TOTL",
    "gdp_per_capita":    "NY.GDP.PCAP.CD",
    "inflation_cpi":     "FP.CPI.TOTL.ZG",
    "unemployment_rate":     "SL.UEM.TOTL.ZS",
    "youth_unemployment":    "SL.UEM.1524.NE.ZS",

}

rows = []

for indicator_name, indicator_code in INDICATORS.items():

    # Build URL: all countries batched in a single request per indicator
    url = f"{world_bank_url}/{countries}/indicator/{indicator_code}"

    # Request 2024 data in JSON format, per_page=100 fits all 10 countries
    bank_request = requests.get(url, params={"format": "json", "date": "2024", "per_page": 100}, timeout=30)
    bank_request.raise_for_status()  # raises an error if the request failed

    # World Bank response is a list of 2 elements: [metadata, data]
    bank_data = bank_request.json()
    
    # We check len > 1 to confirm the data element exists, and data[1] to confirm it's not empty
    if len(bank_data) > 1 and bank_data[1]:
        for entry in bank_data[1]:
            if entry["value"] is not None:  # skip missing values
                rows.append({
                    "country":   entry["country"]["value"],  # full country name
                    "indicator": indicator_name,
                    "value":     entry["value"]
                })

# Build a tidy dataframe and pivot to wide format (countries as rows, indicators as columns)
world_bank_df = pd.DataFrame(rows)
world_bank_pivot = world_bank_df.pivot_table(
    index="country",
    columns="indicator",
    values="value",
    aggfunc="first"
)

In [3]:
world_bank_pivot

indicator,gdp_per_capita,inflation_cpi,population,unemployment_rate,youth_unemployment
country,,,,,
Austria,58268.878765,2.937916,9177982.0,5.200,10.864
Belgium,56614.567950,3.143491,11858610.0,5.700,17.418
France,46103.084086,1.999049,68551653.0,7.400,19.342
Germany,56103.732318,2.256498,83516593.0,3.400,6.910
Italy,40385.341396,0.982373,58952704.0,6.500,20.316
Netherlands,67520.421896,3.347543,17993485.0,3.700,8.683
Poland,25103.565661,3.790609,36559233.0,2.807,10.656
Spain,35326.768307,2.774178,48848840.0,11.400,26.519
Switzerland,103998.186686,1.062340,9005582.0,4.343,8.231
